[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/20.%20The%20SLAP%20for%20Out-of-Gauge%20%28OOG%29%20Containers%20Problem/P20-Tier-3_executed.ipynb)

---


# 20. The SLAP for Out-of-Gauge (OOG) Containers Problem

## Tier 3: The Sine Cosine Algorithm

### Purpose and scope

This notebook uses the Sine Cosine Algorithm (SCA), a population-based metaheuristic, to search for assignments of out-of-gauge containers to vessel slots. Each candidate assignment is evaluated with a scalar fitness that combines assignment cost, penalties, and a stability reward. The algorithm returns its best candidate; because it is stochastic, that candidate is not guaranteed to be globally optimal.

The input is a small instance with container and slot attributes, assignment costs, stability scores, bay weight limits, and penalty weights. The outputs include the best assignment found, its fitness components, a convergence history, and a parameter-sensitivity experiment. The notebook also includes a cost-only assignment as a reference point; that reference is not an optimality certificate for the penalized model.

### Model assumptions

- Each container should be assigned to exactly one slot, and each slot can hold at most one container.
- A container-slot assignment is physically admissible only when the container's weight and height do not exceed the slot's limits.
- The total assigned weight in each bay must not exceed that bay's limit.
- Assignment cost is minimized, while a larger stability score is preferred. The model combines these preferences into one scalar fitness using the specified weights.
- Stability below the specified threshold receives a soft penalty. It is not itself a physical-feasibility constraint.
- The model does not represent all real stowage considerations, such as lashing interactions between containers, detailed vessel trim and stress calculations, or uncertainty in operational data.

### Search outline

1. Initialize a population of candidate assignments.
2. Update each candidate's continuous position using the SCA sine-or-cosine movement rule toward the best candidate found so far.
3. Convert each updated position into a discrete assignment by selecting the highest-valued slot position for each container.
4. Evaluate the assignment using cost, stability, and penalties; retain an updated candidate only when its fitness improves.
5. Record the best and average population fitness at each iteration.

The initial assignments have one slot per container and do not reuse slots, but they are not necessarily feasible: weight, height, and bay limits are checked by the fitness evaluation. The algorithm runs for the requested number of iterations; it does not implement an early stopping rule based on stagnation.


## Mathematical model and SCA representation

A binary decision variable records whether container \(i\) is assigned to slot \(j\):

$$
x_{ij} =
\begin{cases}
1, & \text{if container } i \text{ is assigned to slot } j,\\
0, & \text{otherwise.}
\end{cases}
$$

Let \(C_{ij}\) be the assignment cost and \(S_{ij}\) the stability score for that pair. The total cost and stability score are

$$
C(x) = \sum_i \sum_j C_{ij}x_{ij},
\qquad
S(x) = \sum_i \sum_j S_{ij}x_{ij}.
$$

The fitness minimized by SCA is

$$
F(x) = C(x) + P(x) - 10S(x),
$$

where \(P(x)\) is the sum of the configured penalties. It includes penalties for unassigned or multiply assigned containers, slot conflicts, weight and height violations, bay-limit violations, and stability below the threshold. The stability threshold contributes a soft penalty; feasibility is determined by assignment and physical constraints, not by that soft penalty. In the code, `calculate_fitness` computes \(F\), its cost and penalty components, and \(S\); `is_feasible_assignment` checks the hard assignment and physical constraints.

For the SCA movement, a candidate is represented by a continuous position vector \(X\), while the best candidate is \(X^*\). At iteration \(t\), the control value is \(r_1=a(1-t/T)\), where \(a\) is the initial control parameter and \(T\) is the maximum iteration count. Random values \(r_2\), \(r_3\), and \(r_4\) select and scale the movement:

$$
X_{t+1} = X_t + r_1
\begin{cases}
\sin(r_2)\left|r_3X^*-X_t\right|, & r_4 < 0.5,\\
\cos(r_2)\left|r_3X^*-X_t\right|, & r_4 \geq 0.5.
\end{cases}
$$

Here \(r_2\) is sampled from \([0,2\pi]\), \(r_3\) from \([0,2]\), and \(r_4\) from \([0,1]\). The updated vector is reshaped into a container-by-slot matrix. For each container, the slot with the largest position value is selected; this discretization can create slot conflicts, which the fitness function penalizes.

SCA does not use a Bellman recurrence or enumerate all assignments. Its work grows with the population size, iteration count, and number of position coordinates: for \(n\) containers and \(m\) slots, each iteration updates \(N\) candidates with \(nm\) coordinates, in addition to fitness evaluation. The corresponding dominant movement work is proportional to \(TNnm\), where \(T\) is the iteration count and \(N\) is the population size. The code implements the movement in `sine_cosine_position_update`, the discretization in `position_to_assignment`, and the population search in `sine_cosine_algorithm`.


In [ ]:
from dataclasses import dataclass
import math
import time
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Consistent plot styling makes the search history and comparisons easier to read.
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("Libraries imported successfully for Sine Cosine Optimization.")

In [ ]:
@dataclass
class OOGContainer:
    """An out-of-gauge container and its relevant assignment attributes."""

    id: int
    weight: float  # tons
    height: float  # meters
    length: float  # meters
    container_type: str
    lashing_coeff: float


@dataclass
class VesselSlot:
    """A vessel slot with capacity, bay, and crane-distance attributes."""

    id: int
    weight_capacity: float  # tons
    height_limit: float  # meters
    bay_id: int
    crane_distance: float


@dataclass
class SCASolution:
    """A discrete assignment and its evaluated objective components."""

    assignment: np.ndarray
    fitness: float
    cost: float
    penalty: float
    stability: float


@dataclass
class SCAProblem:
    """The assignment data and penalty parameters used by the SCA search."""

    containers: List[OOGContainer]
    slots: List[VesselSlot]
    cost_matrix: np.ndarray
    stability_matrix: np.ndarray
    bay_weight_limits: Dict[int, float]
    penalty_weights: Dict[str, float]


print("Data structures defined for Sine Cosine Optimization.")

In [ ]:
# This instance uses the same containers, slots, and assignment data throughout the notebook.
containers = [
    OOGContainer(1, 45, 2.8, 12.0, "flat rack", 2.1),
    OOGContainer(2, 32, 4.2, 6.0, "open-top", 3.8),
    OOGContainer(3, 38, 2.6, 12.0, "platform", 1.9),
]

slots = [
    VesselSlot(1, 50, 3.0, 2, 15.0),
    VesselSlot(2, 40, 5.0, 3, 12.0),
    VesselSlot(3, 45, 3.0, 3, 18.0),
    VesselSlot(4, 60, 4.0, 3, 10.0),
]

cost_matrix = np.array(
    [
        [100, 150, 120, 90],
        [80, 70, 110, 85],
        [95, 140, 100, 80],
    ],
    dtype=float,
)

# Larger values represent better stability for the corresponding assignment.
stability_matrix = np.array(
    [
        [0.8, 0.7, 0.9, 0.85],
        [0.75, 0.8, 0.7, 0.9],
        [0.85, 0.65, 0.8, 0.75],
    ],
    dtype=float,
)

bay_weight_limits = {2: 80, 3: 120}

penalty_weights = {
    "constraint_violation": 500.0,
    "poor_stability": 100.0,
    "unassigned_container": 1000.0,
}

problem = SCAProblem(
    containers=containers,
    slots=slots,
    cost_matrix=cost_matrix,
    stability_matrix=stability_matrix,
    bay_weight_limits=bay_weight_limits,
    penalty_weights=penalty_weights,
)

n_containers = len(problem.containers)
n_slots = len(problem.slots)
assert problem.cost_matrix.shape == (n_containers, n_slots)
assert problem.stability_matrix.shape == (n_containers, n_slots)
assert {slot.bay_id for slot in problem.slots}.issubset(problem.bay_weight_limits)

print(f"SCA problem defined: {n_containers} containers, {n_slots} slots.")
print(f"Penalty weights: {problem.penalty_weights}")
print(f"Cost matrix shape: {problem.cost_matrix.shape}")

## Candidate generation and fitness evaluation

A candidate assignment is stored as a binary matrix: row \(i\) represents a container and column \(j\) represents a slot. A value of one means that the container is assigned to that slot. The fitness evaluation gives the search a single numerical score to minimize while retaining separate cost, penalty, and stability values for interpretation.

The penalty weights determine how strongly violations influence the search. A penalty discourages an infeasible assignment; it does not mathematically guarantee that every returned candidate is feasible. Use the explicit feasibility check when interpreting a result.

In [ ]:
def generate_random_solution(problem: SCAProblem) -> SCASolution:
    """Create a random one-slot-per-container candidate without slot reuse."""
    n_containers = len(problem.containers)
    n_slots = len(problem.slots)

    assignment = np.zeros((n_containers, n_slots), dtype=int)
    available_slots = np.random.permutation(n_slots)

    for i in range(n_containers):
        if i < n_slots:
            assignment[i, available_slots[i]] = 1

    fitness, cost, penalty, stability = calculate_fitness(problem, assignment)
    return SCASolution(assignment, fitness, cost, penalty, stability)


def is_feasible_assignment(problem: SCAProblem, assignment: np.ndarray) -> bool:
    """Check assignment, slot, individual-capacity, and bay-weight constraints."""
    n_containers = len(problem.containers)
    n_slots = len(problem.slots)

    if assignment.shape != (n_containers, n_slots):
        return False
    if not np.isin(assignment, [0, 1]).all():
        return False
    if not np.all(assignment.sum(axis=1) == 1):
        return False
    if not np.all(assignment.sum(axis=0) <= 1):
        return False

    bay_weights = {bay_id: 0.0 for bay_id in problem.bay_weight_limits}
    for i, container in enumerate(problem.containers):
        for j, slot in enumerate(problem.slots):
            if assignment[i, j] == 1:
                if container.weight > slot.weight_capacity:
                    return False
                if container.height > slot.height_limit:
                    return False
                bay_weights[slot.bay_id] += container.weight

    return all(
        total_weight <= problem.bay_weight_limits[bay_id]
        for bay_id, total_weight in bay_weights.items()
    )


def calculate_fitness(
    problem: SCAProblem, assignment: np.ndarray
) -> Tuple[float, float, float, float]:
    """Return fitness, assignment cost, total penalty, and stability score."""
    n_containers = len(problem.containers)
    n_slots = len(problem.slots)

    if assignment.shape != (n_containers, n_slots):
        raise ValueError("Assignment matrix has an incompatible shape.")
    if not np.isin(assignment, [0, 1]).all():
        raise ValueError("Assignment matrix must contain only binary values.")

    cost = float(np.sum(problem.cost_matrix * assignment))
    stability = float(np.sum(problem.stability_matrix * assignment))
    penalty = 0.0
    constraint_penalty = problem.penalty_weights["constraint_violation"]

    # Row and column counts measure assignment completeness and slot conflicts.
    for i in range(n_containers):
        assigned_count = int(assignment[i].sum())
        if assigned_count == 0:
            penalty += problem.penalty_weights["unassigned_container"]
        elif assigned_count > 1:
            penalty += constraint_penalty * (assigned_count - 1)

        # Check every selected pair so multiple assignments cannot bypass capacity checks.
        for j, slot in enumerate(problem.slots):
            if assignment[i, j] == 1:
                container = problem.containers[i]
                if container.weight > slot.weight_capacity:
                    penalty += constraint_penalty
                if container.height > slot.height_limit:
                    penalty += constraint_penalty

    for j in range(n_slots):
        assigned_count = int(assignment[:, j].sum())
        if assigned_count > 1:
            penalty += constraint_penalty * (assigned_count - 1)

    bay_weights = {bay_id: 0.0 for bay_id in problem.bay_weight_limits}
    for i, container in enumerate(problem.containers):
        for j, slot in enumerate(problem.slots):
            if assignment[i, j] == 1:
                bay_weights[slot.bay_id] += container.weight

    for bay_id, total_weight in bay_weights.items():
        if total_weight > problem.bay_weight_limits[bay_id]:
            penalty += constraint_penalty

    if stability < 2.0:
        penalty += problem.penalty_weights["poor_stability"]

    fitness = cost + penalty - 10.0 * stability
    return fitness, cost, penalty, stability


print("Candidate generation and fitness evaluation functions defined.")

## SCA movement and discrete assignment

SCA moves candidates in a continuous search space, but the stowage decision is discrete. The position vector is therefore reshaped into one row per container, and the largest value in each row selects that container's slot. This row-wise rule guarantees one selected slot per container, but it does not guarantee that different containers select different slots; slot conflicts remain part of the penalized fitness evaluation.

In [ ]:
def sine_cosine_position_update(
    current_position: np.ndarray,
    best_position: np.ndarray,
    a: float,
    iteration: int,
    max_iterations: int,
) -> np.ndarray:
    """Update one continuous candidate using the SCA sine-or-cosine rule."""
    r1 = a * (1.0 - iteration / max_iterations)
    r2 = 2.0 * np.pi * np.random.random()
    r3 = 2.0 * np.random.random()
    r4 = np.random.random()

    distance = np.abs(r3 * best_position - current_position)
    if r4 < 0.5:
        return current_position + r1 * np.sin(r2) * distance
    return current_position + r1 * np.cos(r2) * distance


def position_to_assignment(
    position: np.ndarray, n_containers: int, n_slots: int
) -> np.ndarray:
    """Map a continuous position vector to a binary assignment matrix."""
    position_matrix = position.reshape(n_containers, n_slots)
    assignment = np.zeros((n_containers, n_slots), dtype=int)

    for i in range(n_containers):
        best_slot = int(np.argmax(position_matrix[i]))
        assignment[i, best_slot] = 1

    return assignment


def assignment_to_position(assignment: np.ndarray) -> np.ndarray:
    """Flatten a binary assignment matrix into a continuous position vector."""
    return assignment.astype(float).flatten()


print("Sine Cosine Algorithm position-update functions defined.")

## Population search

The population is a collection of candidate assignments. At each iteration, every candidate is moved relative to the best candidate found so far, converted back to a discrete assignment, and evaluated. A candidate replaces its current population member only if its fitness is lower. Consequently, the best fitness retained by the algorithm cannot increase as the search proceeds.

The random draws make the search stochastic. No random seed is set in this notebook, so separate executions may follow different search paths and return different candidates. The population size, iteration count, and control parameter below determine the search effort and movement scale.

In [ ]:
def sine_cosine_algorithm(
    problem: SCAProblem,
    population_size: int = 25,
    max_iterations: int = 60,
    a: float = 2.0,
) -> Tuple[SCASolution, List[Dict[str, float]]]:
    """Search for a low-fitness assignment using the Sine Cosine Algorithm."""
    if population_size <= 0:
        raise ValueError("Population size must be positive.")
    if max_iterations <= 0:
        raise ValueError("Maximum iterations must be positive.")
    if a < 0:
        raise ValueError("The SCA control parameter must be nonnegative.")

    n_containers = len(problem.containers)
    n_slots = len(problem.slots)

    population = [
        generate_random_solution(problem) for _ in range(population_size)
    ]
    best_solution = min(population, key=lambda solution: solution.fitness)
    convergence_history: List[Dict[str, float]] = []

    for iteration in range(max_iterations):
        for i in range(population_size):
            current_solution = population[i]
            current_position = assignment_to_position(current_solution.assignment)
            best_position = assignment_to_position(best_solution.assignment)

            new_position = sine_cosine_position_update(
                current_position,
                best_position,
                a,
                iteration,
                max_iterations,
            )
            new_assignment = position_to_assignment(
                new_position, n_containers, n_slots
            )

            fitness, cost, penalty, stability = calculate_fitness(
                problem, new_assignment
            )
            new_solution = SCASolution(
                new_assignment, fitness, cost, penalty, stability
            )

            if new_solution.fitness < current_solution.fitness:
                population[i] = new_solution
                if new_solution.fitness < best_solution.fitness:
                    best_solution = new_solution

        average_fitness = float(
            np.mean([solution.fitness for solution in population])
        )
        convergence_history.append(
            {
                "iteration": iteration + 1,
                "best_fitness": best_solution.fitness,
                "avg_fitness": average_fitness,
                "best_cost": best_solution.cost,
                "best_penalty": best_solution.penalty,
                "best_stability": best_solution.stability,
            }
        )

        if (iteration + 1) % 10 == 0:
            print(
                f"Iteration {iteration + 1:3d}: "
                f"Best fitness = {best_solution.fitness:.2f}, "
                f"Cost = {best_solution.cost:.2f}, "
                f"Penalty = {best_solution.penalty:.2f}"
            )

    return best_solution, convergence_history


print("Sine Cosine Algorithm implementation defined.")

## Run the SCA search

This run uses the stated population size, iteration count, and control parameter. The reported fitness is the scalar objective used by the search; cost, penalty, and stability are also shown separately. A zero total penalty indicates that no penalized condition was triggered, while `is_feasible_assignment` independently checks the hard assignment and physical constraints.

In [ ]:
print("SINE COSINE ALGORITHM FOR OOG CONTAINER ASSIGNMENT")
print("=" * 54)
print("Parameters: population size = 25, maximum iterations = 60, a = 2.0")

start_time = time.time()
best_solution, convergence_history = sine_cosine_algorithm(
    problem, population_size=25, max_iterations=60, a=2.0
)
execution_time = time.time() - start_time

is_feasible = is_feasible_assignment(problem, best_solution.assignment)
assert math.isclose(
    best_solution.fitness,
    best_solution.cost + best_solution.penalty - 10.0 * best_solution.stability,
    rel_tol=1e-12,
    abs_tol=1e-12,
)

print(f"\nExecution time: {execution_time:.3f} seconds")
print("\nBEST CANDIDATE FOUND:")
print(f"  Fitness: {best_solution.fitness:.2f}")
print(f"  Total assignment cost: {best_solution.cost:.2f} units")
print(f"  Total penalty: {best_solution.penalty:.2f}")
print(f"  Stability score: {best_solution.stability:.2f}")
print(f"  Hard-constraint feasibility: {'Feasible' if is_feasible else 'Infeasible'}")

print("\nASSIGNMENTS:")
for i, container in enumerate(problem.containers):
    assigned_slots = np.flatnonzero(best_solution.assignment[i])
    for j in assigned_slots:
        slot = problem.slots[j]
        print(
            f"  Container {container.id} ({container.container_type}, "
            f"{container.weight} tons, {container.height} m) -> "
            f"Slot {slot.id} (capacity: {slot.weight_capacity} tons, "
            f"height limit: {slot.height_limit} m, bay: {slot.bay_id})"
        )
        print(
            f"    Assignment cost: {problem.cost_matrix[i, j]:.0f}; "
            f"stability score: {problem.stability_matrix[i, j]:.2f}"
        )

total_assigned_weight = 0.0
total_assigned_capacity = 0.0
bay_weights = {bay_id: 0.0 for bay_id in problem.bay_weight_limits}

for i, container in enumerate(problem.containers):
    for j, slot in enumerate(problem.slots):
        if best_solution.assignment[i, j] == 1:
            total_assigned_weight += container.weight
            total_assigned_capacity += slot.weight_capacity
            bay_weights[slot.bay_id] += container.weight

utilization_rate = (
    100.0 * total_assigned_weight / total_assigned_capacity
    if total_assigned_capacity > 0
    else 0.0
)

print("\nPERFORMANCE METRICS:")
print(f"  Assigned weight / assigned slot capacity: {utilization_rate:.1f}%")
print(f"  Hard-constraint feasibility: {'Feasible' if is_feasible else 'Infeasible'}")
print("\nBAY WEIGHT DISTRIBUTION:")
for bay_id, weight in bay_weights.items():
    limit = problem.bay_weight_limits[bay_id]
    bay_utilization = 100.0 * weight / limit if limit > 0 else 0.0
    print(
        f"  Bay {bay_id}: {weight:.1f} / {limit:.1f} tons "
        f"({bay_utilization:.1f}% of limit)"
    )

## Cost-only assignment reference

The following reference uses the Hungarian algorithm to minimize the assignment-cost matrix alone. It is included as a cost-only point of comparison, not as a second solution method for the penalized SCA model. Its assignment is evaluated with the same fitness and feasibility functions as the SCA candidate, so the comparison makes clear whether the cost-only assignment also satisfies the modeled constraints and how its stability score affects fitness.

In [ ]:
from scipy.optimize import linear_sum_assignment

row_ind, col_ind = linear_sum_assignment(problem.cost_matrix)
reference_assignment = np.zeros((len(containers), len(slots)), dtype=int)
reference_assignment[row_ind, col_ind] = 1

reference_fitness, reference_cost, reference_penalty, reference_stability = (
    calculate_fitness(problem, reference_assignment)
)
reference_feasible = is_feasible_assignment(problem, reference_assignment)

print("COST-ONLY ASSIGNMENT REFERENCE")
print("=" * 38)
print(f"Fitness under the penalized model: {reference_fitness:.2f}")
print(f"Assignment cost: {reference_cost:.2f} units")
print(f"Total penalty: {reference_penalty:.2f}")
print(f"Stability score: {reference_stability:.2f}")
print(f"Hard-constraint feasibility: {'Feasible' if reference_feasible else 'Infeasible'}")

print("\nSCA CANDIDATE")
print(f"Fitness: {best_solution.fitness:.2f}")
print(f"Assignment cost: {best_solution.cost:.2f} units")
print(f"Total penalty: {best_solution.penalty:.2f}")
print(f"Stability score: {best_solution.stability:.2f}")
print(f"Hard-constraint feasibility: {'Feasible' if is_feasible else 'Infeasible'}")

print("\nSCA MINUS COST-ONLY REFERENCE")
print(f"Fitness difference: {best_solution.fitness - reference_fitness:+.2f}")
print(f"Cost difference: {best_solution.cost - reference_cost:+.2f} units")
print(f"Stability difference: {best_solution.stability - reference_stability:+.2f}")

print("\nASSIGNMENT COMPARISON")
for i, container in enumerate(problem.containers):
    reference_slots = np.flatnonzero(reference_assignment[i])
    sca_slots = np.flatnonzero(best_solution.assignment[i])
    reference_slot_id = (
        problem.slots[reference_slots[0]].id if len(reference_slots) else None
    )
    sca_slot_id = problem.slots[sca_slots[0]].id if len(sca_slots) else None
    print(
        f"Container {container.id}: cost-only reference slot "
        f"{reference_slot_id}; SCA slot {sca_slot_id}"
    )

## Convergence and solution-component plots

The first plot tracks the best fitness retained and the average fitness in the population. A falling best-fitness curve indicates that the search has found an improved candidate; the average curve gives context about the population as a whole. The other panels track the best candidate's cost, penalty, and stability, and compare its components with those of the cost-only reference. These plots describe this stochastic run; they do not establish global optimality.

In [ ]:
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(
    "Sine Cosine Algorithm Analysis: OOG Container Assignment",
    fontsize=16,
    fontweight="bold",
)

iterations = [entry["iteration"] for entry in convergence_history]
best_fitness_values = [entry["best_fitness"] for entry in convergence_history]
average_fitness_values = [entry["avg_fitness"] for entry in convergence_history]

ax1.plot(iterations, best_fitness_values, linewidth=2, label="Best fitness")
ax1.plot(
    iterations,
    average_fitness_values,
    linestyle="--",
    linewidth=1,
    label="Average population fitness",
)
ax1.axhline(
    y=reference_fitness,
    linestyle=":",
    linewidth=2,
    label="Cost-only reference evaluated by fitness",
)
ax1.set_xlabel("Iteration")
ax1.set_ylabel("Fitness (lower is better)")
ax1.set_title("Fitness history")
ax1.legend()
ax1.grid(True, alpha=0.3)

cost_values = [entry["best_cost"] for entry in convergence_history]
penalty_values = [entry["best_penalty"] for entry in convergence_history]
ax2_twin = ax2.twinx()
cost_line = ax2.plot(iterations, cost_values, linewidth=2, label="Assignment cost")
penalty_line = ax2_twin.plot(
    iterations,
    penalty_values,
    color="tab:red",
    linewidth=2,
    label="Total penalty",
)
ax2.axhline(
    y=reference_cost,
    color="tab:blue",
    linestyle=":",
    linewidth=2,
    label="Cost-only reference cost",
)
ax2.set_xlabel("Iteration")
ax2.set_ylabel("Assignment cost (units)")
ax2_twin.set_ylabel("Total penalty")
ax2.set_title("Cost and penalty history")
ax2.legend(cost_line + penalty_line, [line.get_label() for line in cost_line + penalty_line])
ax2.grid(True, alpha=0.3)

stability_values = [entry["best_stability"] for entry in convergence_history]
ax3.plot(iterations, stability_values, linewidth=2, label="SCA candidate")
ax3.axhline(
    y=reference_stability,
    color="tab:orange",
    linestyle=":",
    linewidth=2,
    label="Cost-only reference",
)
ax3.set_xlabel("Iteration")
ax3.set_ylabel("Stability score (higher is better)")
ax3.set_title("Stability history")
ax3.legend()
ax3.grid(True, alpha=0.3)

methods = ["Cost-only reference", "SCA candidate"]
fitness_values = [reference_fitness, best_solution.fitness]
comparison_costs = [reference_cost, best_solution.cost]
comparison_stabilities = [reference_stability, best_solution.stability]
x = np.arange(len(methods))
width = 0.25

ax4.bar(x - width, fitness_values, width, label="Fitness")
ax4.bar(x, comparison_costs, width, label="Assignment cost")
ax4.bar(x + width, comparison_stabilities, width, label="Stability score")
ax4.set_xlabel("Candidate")
ax4.set_ylabel("Value (fitness and cost in model units)")
ax4.set_title("Evaluated solution components")
ax4.set_xticks(x)
ax4.set_xticklabels(methods, rotation=10, ha="right")
ax4.legend()
ax4.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("Convergence and solution-component plots created.")

## Parameter sensitivity experiment

SCA's search behavior depends on population size, iteration count, and the control parameter \(a\). The scenarios below vary these settings while keeping the problem data and penalty weights unchanged. Because each run uses random draws and no seed is fixed, a scenario's result is one stochastic observation, not a guarantee about its average performance.

The reported attainment iteration is the first recorded iteration at which the run's final best fitness is reached, using a floating-point tolerance. This describes when the final value appeared in that run; it is not a separate stopping rule or proof of convergence.

In [ ]:
parameter_scenarios = [
    {"name": "Base case", "pop_size": 25, "max_iter": 60, "a": 2.0},
    {"name": "Large population", "pop_size": 50, "max_iter": 60, "a": 2.0},
    {"name": "More iterations", "pop_size": 25, "max_iter": 100, "a": 2.0},
    {"name": "High exploration", "pop_size": 25, "max_iter": 60, "a": 3.0},
    {"name": "Balanced settings", "pop_size": 30, "max_iter": 80, "a": 2.5},
]

sensitivity_results = []

for scenario in parameter_scenarios:
    print(f"\nTesting: {scenario['name']}")
    print(
        f"  Parameters: population size = {scenario['pop_size']}, "
        f"maximum iterations = {scenario['max_iter']}, a = {scenario['a']}"
    )

    start_time = time.time()
    solution, history = sine_cosine_algorithm(
        problem,
        population_size=scenario["pop_size"],
        max_iterations=scenario["max_iter"],
        a=scenario["a"],
    )
    scenario_time = time.time() - start_time

    attainment_iteration = next(
        (
            int(entry["iteration"])
            for entry in history
            if math.isclose(
                entry["best_fitness"],
                solution.fitness,
                rel_tol=1e-9,
                abs_tol=1e-9,
            )
        ),
        len(history),
    )

    sensitivity_results.append(
        {
            "scenario": scenario["name"],
            "fitness": solution.fitness,
            "cost": solution.cost,
            "penalty": solution.penalty,
            "stability": solution.stability,
            "iterations": len(history),
            "execution_time": scenario_time,
            "attainment_iteration": attainment_iteration,
        }
    )

    print(
        f"  Result: fitness = {solution.fitness:.2f}, "
        f"cost = {solution.cost:.2f}, "
        f"time = {scenario_time:.3f} seconds"
    )

scenario_names = [result["scenario"] for result in sensitivity_results]
fitnesses = [result["fitness"] for result in sensitivity_results]
costs = [result["cost"] for result in sensitivity_results]
times = [result["execution_time"] for result in sensitivity_results]
attainment_iterations = [
    result["attainment_iteration"] for result in sensitivity_results
]

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle("SCA Parameter Sensitivity: Single-Run Results", fontsize=16)

ax1.bar(scenario_names, fitnesses, alpha=0.8, color="lightblue")
ax1.axhline(
    y=reference_fitness,
    color="red",
    linestyle="--",
    linewidth=2,
    label="Cost-only reference evaluated by fitness",
)
ax1.set_ylabel("Fitness (lower is better)")
ax1.set_title("Fitness by configuration")
ax1.tick_params(axis="x", rotation=35)
ax1.legend()
ax1.grid(True, axis="y", alpha=0.3)

ax2.bar(scenario_names, costs, alpha=0.8, color="lightgreen")
ax2.axhline(
    y=reference_cost,
    color="red",
    linestyle="--",
    linewidth=2,
    label="Cost-only reference",
)
ax2.set_ylabel("Assignment cost (units)")
ax2.set_title("Assignment cost by configuration")
ax2.tick_params(axis="x", rotation=35)
ax2.legend()
ax2.grid(True, axis="y", alpha=0.3)

ax3.bar(scenario_names, times, alpha=0.8, color="lightcoral")
ax3.set_ylabel("Execution time (seconds)")
ax3.set_title("Execution time by configuration")
ax3.tick_params(axis="x", rotation=35)
ax3.grid(True, axis="y", alpha=0.3)

ax4.bar(scenario_names, attainment_iterations, alpha=0.8, color="gold")
ax4.set_ylabel("Iteration")
ax4.set_title("First attainment of each run's final best fitness")
ax4.tick_params(axis="x", rotation=35)
ax4.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("\nInterpretation notes:")
print("  Population size and iteration count affect search effort and runtime.")
print("  The control parameter changes the scale of SCA position updates.")
print("  These stochastic single-run results do not establish average performance.")

## When this approach is useful

SCA is a population-based search method that can be applied when a problem is difficult to optimize directly and a useful candidate is more important than a proof of global optimality. Its trigonometric movement rules balance larger and smaller changes through a control parameter that decreases over the run. For this assignment model, the search operates on a scalar fitness; it does not independently optimize multiple objectives or return a Pareto frontier.

The method has important limitations. Its results depend on stochastic search and parameter settings, penalty weights can influence which trade-offs are favored, and a finite run can miss better assignments. For a small instance, the cost-only reference is useful for context but does not solve the full penalized model. The feasibility check remains necessary when the operational requirement is a valid assignment.

The same modeling structure can be extended with additional assignment constraints or a larger input instance, provided the fitness evaluation and feasibility checks are updated consistently. Such extensions change the problem being modeled and should be stated explicitly.